# Day 04 上午课堂练习：电商用户数据清洗与预处理

**主数据文件：** E Commerce Dataset.xlsx（使用 E Comm 工作表）

**提交要求：** 完成所有 TODO，运行全部单元后提交本 Notebook 与清洗后的 CSV 文件。

## 学习目标

- 检查字段类型、缺失值和重复记录；
- 使用中位数填补数值缺失；
- 统一类别字段的同义取值；
- 使用统计规则和业务规则检查候选异常值；
- 导出清洗后的数据。

---
## 1. 读取数据

如报找不到文件，请修改候选路径或 DATA_PATH。

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("/Users/yq/muc_training/data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

df = pd.read_excel(DATA_PATH, sheet_name="E Comm")
print(f"读取文件：{DATA_PATH}")
print(f"数据形状：{df.shape[0]} 行，{df.shape[1]} 列")
df.head()

### 任务 1：理解数据

运行下一单元，并以注释回答：

1. 一行数据代表什么？
2. 哪个字段是用户唯一标识？
3. 哪个字段可作为用户流失分析的目标变量？

In [ ]:
df.info()

# 答案：
# 1.一行数据代表一个具体的用户样本
# 2.CustomerID是用户唯一标识
# 3.Churn可作为用户流失分析的目标变量

---
## 2. 数据质量检查

数据清洗前，先检查缺失值和重复值。

### 任务 2：生成缺失值报告

创建 missing_report，包含“缺失数量”和“缺失比例”两列；按缺失数量降序排列。缺失比例用百分比表示，保留两位小数。

In [ ]:
# TODO：创建 missing_report
# 提示：df.isna().sum() 统计缺失数量；df.isna().mean() 计算缺失比例。
missing_report = df.isnull().sum().sort_values(ascending=False)

present_report = (df.isnull().mean()*100).round(1).sort_values(ascending=False)

print(f'缺失数量:\n{missing_report}')
print(f'缺失比例:\n{present_report}')

### 任务 3：检查重复记录

分别统计完全重复行数与 CustomerID 重复数量。思考：CustomerID 重复时，能否直接删除？

In [ ]:
# TODO：完成两项重复值统计
# 1.完全重复的行数统计
duplicate_rows = df.duplicated().sum()
print(f"重复的行数: {duplicate_rows}")
# 2.CustomerID的重复数量
duplicate_customer_ids = df['CustomerID'].value_counts().max() - 1
print("CustomerID 重复数量：", duplicate_customer_ids)

# 思考：用户 ID 重复时，不能直接删除，因为有可能记录了用户不同时间点的状态，并根据业务逻辑选择保留哪一条。

---
## 3. 缺失值处理

本练习对数值型缺失统一采用中位数填充。缺失不等于 0，例如订单数缺失并不能说明用户没有下单。

### 任务 4：用中位数填补数值缺失

请对下列字段逐列使用中位数填充，随后检查剩余缺失值。

In [ ]:
numeric_missing_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

# TODO：循环填充每列的中位数
for col in numeric_missing_cols:
    df[col] = df[col].fillna(df[col].median())

# TODO：输出上述字段剩余的缺失值数量
print(df[numeric_missing_cols].isna().sum())

### 思考题

什么时候不适合用中位数填充？写出一种场景及更合适的处理思路。

In [ ]:
# 场景：例如使用优惠券的场景，有些用户没有使用过，如果直接用中位数填充这会导致将这些用户标记未“活跃优惠券使用者”，会整体拉高优惠券使用率的统计值，最终严重影响后续的营销策略，比如给这些本不该发券的人发券，造成成本浪费。
# 处理思路：应该用0来填充，因为“没有记录”在计数类字段中通常等同于“次数为0”

---
## 4. 类别字段标准化

同一业务含义被写成不同文本，会导致统计结果被拆散。先观察，再统一；不要在没有业务依据的情况下随意合并。

### 任务 5：查看类别取值

检查登录设备、支付方式和订单品类字段，记录你发现的同义类别。

In [ ]:
category_cols = [
    "PreferredLoginDevice",
    "PreferredPaymentMode",
    "PreferedOrderCat",
]

for col in category_cols:
    print(f"\n{col}")
    print(df[col].value_counts())

# PreferredLoginDevice中 Mobile Phone 和 Phone 是同义的
# PreferredPaymentMode中 COD 和 Cash on Delivery 是同义的
# PreferedOrderCat 中 Mobile Phone 和 Mobile 是同义的

### 任务 6：统一同义类别

按以下规则进行标准化：

- Phone → Mobile Phone
- COD → Cash on Delivery
- CC → Credit Card
- Mobile → Mobile Phone

处理后重新输出频数。

In [26]:
# TODO：完成类别标准化
df["PreferredLoginDevice"] = df['PreferredLoginDevice'].replace({'Phone': 'Mobile Phone'})
df["PreferredPaymentMode"] = df['PreferredPaymentMode'].replace({'COD': 'Cash on Delivery'})
df['PreferredPaymentMode'] = df['PreferredPaymentMode'].replace({'CC': 'Credit Card'})
df["PreferedOrderCat"] = df['PreferedOrderCat'].replace({'Mobile': 'Mobile Phone'})

# TODO：重新检查类别频数
for col in category_cols:
    print(f"\n{col}")
    print(df[col].value_counts())


PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    3996
Computer        1634
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1774
E wallet             614
Cash on Delivery     514
UPI                  414
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Mobile Phone          2080
Laptop & Accessory    2050
Fashion                826
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 5. 候选异常值检查

IQR 方法只能发现候选异常值，不能直接证明数据错误。处理前必须结合业务判断。

In [ ]:
def iqr_outlier_summary(series):
    """返回数值字段的 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return pd.Series({
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": ((series < lower) | (series > upper)).sum()
    })

### 任务 7：检查候选异常值

分别检查 WarehouseToHome、OrderCount 和 CashbackAmount。随后说明：候选异常值能否直接删除，为什么？

In [ ]:
# TODO：调用函数检查三个字段
display1 = iqr_outlier_summary(df["WarehouseToHome"])
display2 = (iqr_outlier_summary(df["OrderCount"]))
display3 = (iqr_outlier_summary(df["CashbackAmount"]))
print('display1:',display1)
print('display2:',display2)
print('display3:',display3)

# 结论：候选异常值不能直接删除，对于 OrderCount—— 删除会丢失核心用户;对于 CashbackAmount—— 删除会破坏消费能力分层;而对于WarehouseToHome，只有两个异常值，可能是录入的时候出现错误，可以单独拎出来查看。如果是明显的脏数据，可以修正或剔除；如果是真实存在的远距离用户，应保留。

### 任务 8：业务规则检查

统计以下不符合业务规则的记录数量：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

In [23]:
# TODO：完成业务规则检查
rules = {
    "使用时长小于 0": (df['Tenure']<0).sum(),
    "仓库距离小于 0": (df['WarehouseToHome']<0).sum(),
    "订单数小于或等于 0": (df['OrderCount']<=0).sum(),
    "返现金额小于 0": (df['CashbackAmount']<0).sum(),
}
pd.Series(rules)

使用时长小于 0      0
仓库距离小于 0      0
订单数小于或等于 0    0
返现金额小于 0      0
dtype: int64

---
## 6. 清洗结果验收与导出

在导出前确认：指定数值字段不再有缺失；类别同义值已统一；输出目录存在。

In [27]:
# TODO：完成验收
assert df[numeric_missing_cols].isna().sum().sum() == 0, "数值字段仍有缺失值"
assert "Phone" not in df["PreferredLoginDevice"].unique(), "登录设备尚未统一"
assert "COD" not in df["PreferredPaymentMode"].unique(), "支付方式尚未统一"
assert "CC" not in df["PreferredPaymentMode"].unique(), "支付方式尚未统一"

print("数据清洗验收通过。")

数据清洗验收通过。


### 任务 9：导出清洗后的数据

将文件导出至 output/ecommerce_customer_cleaned.csv。请确保原始数据不会被覆盖。

In [28]:
OUTPUT_PATH = Path("./output/ecommerce_customer_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# TODO：导出为 UTF-8-SIG 编码的 CSV 文件
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"已导出：{OUTPUT_PATH.resolve()}")

已导出：E:\py_project\东软实习\day04\output\ecommerce_customer_cleaned.csv


---
## 7. 提交前自查

- [ ] 已完成缺失值报告；
- [ ] 已完成重复记录检查；
- [ ] 已填补指定数值字段的缺失值；
- [ ] 已统一登录设备、支付方式和订单品类；
- [ ] 已完成候选异常值与业务规则检查；
- [ ] 已导出 ecommerce_customer_cleaned.csv；
- [ ] 已在关键代码处保留注释，说明处理理由。

## 课后思考

若要基于本数据预测用户流失，哪些字段可以作为特征？CustomerID 是否应该用于建模？为什么？

答：用户属性	Tenure, CityTier, Gender, MaritalStatus, NumberOfAddress, NumberOfDeviceRegistered；
行为特征	HourSpendOnApp, OrderCount, CouponUsed, DaySinceLastOrder, OrderAmountHikeFromlastYear；
服务体验	SatisfactionScore, Complain；
其他	WarehouseToHome, PreferredLoginDevice, PreferredPaymentMode, PreferedOrderCat, CashbackAmount
可作为特征。

CustomerID不应该用于建模，CustomerID 只是一个唯一标识，用来区分不同用户，用来建模没有任何意义